# Tutorial: Basic STREAM data access

This tutorial introduces the public module-level `streamwq` interface. It shows how to discover sites, read a small water-quality and streamflow subset, retrieve catchment attributes, and combine the results.

The client will request your HydroShare credentials when it is created. This notebook reads remote Parquet data without downloading the complete STREAM resource.

## 1. Connect to HydroShare

In [ ]:
import streamwq as stream
from streamwq.joins import join_timeseries

List the dataset names recognized by the client.

In [ ]:
stream.catalog.DATASETS

## 2. Find suitable monitoring sites

Discover sites that advertise either **water temperature** or **dissolved oxygen**, then use metadata to retain sites that report both variables. A site-discovery result identifies candidates and does not guarantee observations in a later date or quality filter.

In [ ]:
candidate_sites = stream.get_sites(variables=["WTemp_C", "DO_mgL"])
candidate_metadata = stream.get_metadata(
    site_ids=candidate_sites,
    columns=["STREAM_ID", "site_name", "state_name", "WQ_parameters"],
).to_pandas()
requested_variables = {"WTemp_C", "DO_mgL"}
available_variables = candidate_metadata["WQ_parameters"].fillna("").map(
    lambda values: {value.strip() for value in values.split(",") if value.strip()}
)
site_metadata = candidate_metadata.loc[
    available_variables.map(requested_variables.issubset)
].reset_index(drop=True)
if site_metadata.empty:
    raise ValueError("No candidate site reports both water temperature and dissolved oxygen")
print(f"Found {len(site_metadata):,} sites with both requested variables")
site_metadata[["STREAM_ID", "site_name", "state_name"]].head(10)

For a reproducible example, use a site known to contain these variables and **streamflow**. If that site is unavailable in a future resource version, use the first matching site instead.

In [ ]:
example_site = site_metadata.iloc[0]["STREAM_ID"]
print(f"Using {example_site} for the rest of the tutorial")

## 3. Read water-quality observations

First, retrieve the complete water-quality record for the selected site and variables. We can then examine the earliest and latest timestamps to determine the period when observations are available.

In [ ]:
water_quality = stream.get_data(
    "water_quality",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "WTemp_C", "DO_mgL"],
)
water_quality.to_pandas()

Read one month rather than the full record. The query selects the station partition, date range, variables, and approved quality flags before converting the result to pandas.

In [ ]:
water_quality = stream.get_data(
    "water_quality",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "WTemp_C", "DO_mgL"],
    start="2022-02-10",
    end="2022-03-10 23:59:59",
    quality="approved",
).to_pandas()
if water_quality.empty:
    raise ValueError("No approved water-quality rows matched this site and date range")
print(f"Retrieved {len(water_quality):,} water-quality records")
water_quality.head()

## 4. Read streamflow observations

Use the same station, dates, and quality policy so the two hourly datasets can be aligned.

In [ ]:
streamflow = stream.get_data(
    "streamflow",
    site_ids=example_site,
    columns=["STREAM_ID", "DateTime", "Q_m3s"],
    start="2022-02-10",
    end="2022-03-10 23:59:59",
    quality="approved",
).to_pandas()
print(f"Retrieved {len(streamflow):,} streamflow records")
streamflow.head()

## 5. Read static catchment attributes

Static attributes have one row per site. Here we request stream order, mean elevation, forest cover, and urban cover.

In [ ]:
attributes = stream.get_data(
    "static_catchment_characteristics",
    site_ids=example_site,
    columns=["STREAM_ID", "stream_order", "ele_mt_sav", "for_pc_sse", "urb_pc_sse"],
).to_pandas()
attributes

## 6. Combine the datasets

Match hourly water quality and streamflow using exact `STREAM_ID` and `DateTime` values, then attach the site-level catchment attributes. `join_timeseries()` uses pandas internally but first protects against missing keys and duplicate station/timestamp rows that could silently produce an incorrect many-to-many merge. It also gives a clear error when periods do not overlap or no timestamps match exactly.

In [ ]:
combined_timeseries = join_timeseries(
    water_quality,
    streamflow,
    suffixes=("_wq", "_flow"),
)

combined = combined_timeseries.merge(
    attributes,
    on="STREAM_ID",
    how="left",
)

print(f"Combined dataset contains {len(combined):,} rows")
combined.head()

## 7. Inspect and optionally export the subset

In [ ]:
combined[["WTemp_C", "DO_mgL", "Q_m3s"]].describe()

In [ ]:
# Uncomment to save the analysis-ready subset.
# combined.to_parquet("stream_basic_subset.parquet", index=False)

## Next steps

You can change the site filters, variables, time period, and accepted quality flags while keeping the same workflow. Later tutorials will cover spatial selection, temporal summaries, and event-based analysis.